In [ ]:
print("all ok")

all ok


In [ ]:
!pip install langchain-community
!pip install pypdf
!pip install python-docx
!pip install pandas
!pip install openpyxl
!pip install beautifulsoup4
!pip install docx2txt
!pip install jq
!pip install unstructured

In [ ]:
!pip install langchain-text-splitters
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetr

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/tmp/ipykernel_4190/3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [ ]:
PDF_PATH = "/content/data"

OUTPUT_DIR = Path("/content/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PDF exists:", os.path.exists(PDF_PATH))

PDF exists: True


In [ ]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("/content/data")

Found 3 PDF files to process

Processing: sample_document.pdf
  ✓ Loaded 1 pages

Processing: complex_rag_parsing_sample_with_sunny_image.pdf
  ✓ Loaded 24 pages

Processing: gen-ai-interview-question-answer.pdf
  ✓ Loaded 187 pages

Total documents loaded: 212


In [ ]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-04T16:55:50+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-04T16:55:50+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/content/data/sample_document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sample_document.pdf', 'file_type': 'pdf'}, page_content='LangChain PDF Loader Sample\nThis PDF is used to demonstrate PyPDFLoader in LangChain.\nRefund Policy\nRefunds are allowed within 7 days of purchase if the product is unused and the invoice is available.\nSupport SLA\nPriority 1 tickets should receive a first response within 2 hours.\nPolicy\nOwner\nUpdate Frequency\nRefund Policy\nSupport\nQuarterly\nLeave Policy\nHR\nYearly'),
 Document(metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': '/content/data/complex_rag_parsing_sample_with_sunny_image.p

In [ ]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [ ]:
chunks=split_documents(all_pdf_documents)
chunks

Split 212 documents into 621 chunks

Example chunk:
Content: LangChain PDF Loader Sample
This PDF is used to demonstrate PyPDFLoader in LangChain.
Refund Policy
Refunds are allowed within 7 days of purchase if the product is unused and the invoice is available....
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-04T16:55:50+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-04T16:55:50+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/content/data/sample_document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sample_document.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-04T16:55:50+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-04T16:55:50+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/content/data/sample_document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sample_document.pdf', 'file_type': 'pdf'}, page_content='LangChain PDF Loader Sample\nThis PDF is used to demonstrate PyPDFLoader in LangChain.\nRefund Policy\nRefunds are allowed within 7 days of purchase if the product is unused and the invoice is available.\nSupport SLA\nPriority 1 tickets should receive a first response within 2 hours.\nPolicy\nOwner\nUpdate Frequency\nRefund Policy\nSupport\nQuarterly\nLeave Policy\nHR\nYearly'),
 Document(metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': '/content/data/complex_rag_parsing_sample_with_sunny_image.p

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
  """Handles document embedding generation using SentenceTransformer"""
  def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
    self.model_name = model_name
    self.model = None
    self._load_model()

  def _load_model(self):
    print(f"Loading embedding model: {self.model_name}")
    self.model = SentenceTransformer(self.model_name)
    print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")

  def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_4190/1194459077.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
      """Initialize ChromaDB client and collection"""
      try:
        # Create persistent ChromaDB client
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        self.collection = self.client.get_or_create_collection(name=self.collection_name)
        print(f"Vector store initialized successfully. Collection name: {self.collection_name}")
      except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Append document text
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized successfully. Collection name: pdf_documents


In [ ]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-04T16:55:50+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-04T16:55:50+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/content/data/sample_document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sample_document.pdf', 'file_type': 'pdf'}, page_content='LangChain PDF Loader Sample\nThis PDF is used to demonstrate PyPDFLoader in LangChain.\nRefund Policy\nRefunds are allowed within 7 days of purchase if the product is unused and the invoice is available.\nSupport SLA\nPriority 1 tickets should receive a first response within 2 hours.\nPolicy\nOwner\nUpdate Frequency\nRefund Policy\nSupport\nQuarterly\nLeave Policy\nHR\nYearly'),
 Document(metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': '/content/data/complex_rag_parsing_sample_with_sunny_image.p

In [ ]:
### Convert the text to embeddings
texts = [ doc.page_content for doc in chunks]

texts

['LangChain PDF Loader Sample\nThis PDF is used to demonstrate PyPDFLoader in LangChain.\nRefund Policy\nRefunds are allowed within 7 days of purchase if the product is unused and the invoice is available.\nSupport SLA\nPriority 1 tickets should receive a first response within 2 hours.\nPolicy\nOwner\nUpdate Frequency\nRefund Policy\nSupport\nQuarterly\nLeave Policy\nHR\nYearly',
 'Complex RAG Parsing Sample - synthetic document\nPage 1\n Complex Document for RAG Parsing Tests\nSynthetic 15-page PDF with paragraphs, simple and complex tables, diagrams, scanned-form style image, metadata\nexamples, and production RAG edge cases.\nStory Line\nThree client teams - Arka Finance, BlueLeaf Retail, and CityRide Mobility - are migrating contracts, policies, support records,\nand operational reports into a single RAG platform. Each team has different document types, access rules, and parsing\nchallenges. The RAG system must answer questions with citations while ensuring that one client never se

In [ ]:
## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)
embeddings

Generating embeddings for 621 texts...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Generated embeddings with shape: (621, 384)


array([[-0.11089499,  0.00705996,  0.02840118, ..., -0.06048666,
         0.04711243,  0.03291167],
       [-0.10510896,  0.06179556, -0.02164228, ..., -0.06212198,
         0.05488476,  0.06049139],
       [-0.03805801,  0.06448637, -0.00823803, ..., -0.03163069,
         0.01721284,  0.01813243],
       ...,
       [ 0.04338337, -0.10247684,  0.03744326, ...,  0.01603347,
        -0.0621482 , -0.03162342],
       [-0.04705645, -0.01850663, -0.00192085, ...,  0.01767375,
        -0.03010236, -0.00323982],
       [-0.06837162,  0.02151505,  0.00442802, ...,  0.00016734,
         0.0179722 ,  0.00347027]], dtype=float32)

In [ ]:
vectorstore.add_documents(chunks,embeddings)

Adding 621 documents to vector store...
Successfully added 621 documents to vector store
Total documents in collection: 621


ChromaDB returns:



results = {
  
    "ids": [
        ["doc_1", "doc_2", "doc_3"]
    ],
    "documents": [
        [
            "Python is a programming language...",
            "Java is object oriented...",
            "Spark is used for big data..."
        ]
    ],
    "metadatas": [
        [
            {"page":1},
            {"page":5},
            {"page":9}
        ]
    ],
    "distances": [
        [0.08, 0.18, 0.42]
    ]
}

|  Threshold | Typical behavior                                                                |
| ---------: | ------------------------------------------------------------------------------- |
|      0.90+ | Very strict; only nearly identical matches are kept.                            |
|  0.80–0.90 | Good starting range for many RAG systems.                                       |
|  0.70–0.80 | More permissive; useful if relevant documents are being missed.                 |
| Below 0.70 | Often includes more unrelated documents, though this varies by embedding model. |


In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Retrieve documents from vector store
        try:
          results = self.vector_store.collection.query(
              query_embeddings=[query_embedding.tolist()],
              n_results=top_k
          )

          print(f"Retrieved {len(results['documents'][0])} documents")

          # Process results
          retrieved_docs = []

          if results['documents'] and results['documents'][0]:
            print(results)
            documents = results['documents'][0]
            metadatas = results['metadatas'][0]
            distances = results['distances'][0]
            ids = results['ids'][0]

            for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
              # Convert distance to similarity score (ChromaDB uses cosine distance)
              similarity_score = 1 - distance
              if similarity_score >= score_threshold:
                retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                print(retrieved_docs)
              else:
                  print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

User Query
     │
     ▼
Generate Query Embedding
     │
     ▼
ChromaDB Query
     │
     ▼
Raw Results

──────────────────────────────────────────

IDs        : [doc1, doc2, doc3]

Documents  : [Python, Java, Spark]

Metadata   : [page1, page5, page9]

Distances  : [0.08, 0.18, 0.42]
──────────────────────────────────────────

     │
     ▼
Loop through each result
     │
     ▼

Compute Similarity = 1 - Distance
     │
     ▼

Threshold Check (e.g., >= 0.80)
     │
     ├── Keep doc1 (0.92)
     ├── Keep doc2 (0.82)
     └── Discard doc3 (0.58)
     │
     ▼

Build `retrieved_docs`
     │
     ▼

Return filtered documents

In [ ]:
rag_retriever

In [ ]:
rag_retriever.retrieve("What is AI")

Retrieving documents for query: 'What is AI'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents
{'ids': [['doc_d89aa9c6_332', 'doc_454d802c_331', 'doc_55a1ec60_329', 'doc_ad657f31_558', 'doc_47896496_38']], 'embeddings': None, 'documents': [['MD  ARBAAZ  KHAN                                                                Generative  AI  Interview  Questions   91 \n \n 4.  Creative  Collaboration:  AI  will  assist  creatives  by  blending  sketches,  descriptions,  and  \nideas\n \ninto\n \ncohesive\n \noutputs,\n \nboosting\n \nartistic\n \nproductivity.\n  5.  Increased  Accessibility:  Multimodal  AI  will  adapt  interactions  to  individual  abilities,  \nimproving\n \naccessibility\n \nfor\n \npeople\n \nwith\n \ndisabilities.\n  6.  Smarter  Autonomous  Systems:  Enhanced  sensor  integration  will  allow  safer  and  more  \nprecise\n \nnavigation\n \nfor\n \nautonomous\n \nvehicles\n \nand\n \ndrones.\n  7.  Data  Integration  Across  Platforms:  AI  will  reduce  data  silos,  combining  information  from  

[{'id': 'doc_d89aa9c6_332',
  'content': 'MD  ARBAAZ  KHAN                                                                Generative  AI  Interview  Questions   91 \n \n 4.  Creative  Collaboration:  AI  will  assist  creatives  by  blending  sketches,  descriptions,  and  \nideas\n \ninto\n \ncohesive\n \noutputs,\n \nboosting\n \nartistic\n \nproductivity.\n  5.  Increased  Accessibility:  Multimodal  AI  will  adapt  interactions  to  individual  abilities,  \nimproving\n \naccessibility\n \nfor\n \npeople\n \nwith\n \ndisabilities.\n  6.  Smarter  Autonomous  Systems:  Enhanced  sensor  integration  will  allow  safer  and  more  \nprecise\n \nnavigation\n \nfor\n \nautonomous\n \nvehicles\n \nand\n \ndrones.\n  7.  Data  Integration  Across  Platforms:  AI  will  reduce  data  silos,  combining  information  from  \nvarious\n \nsources\n \nfor\n \na\n \nmore\n \nholistic\n \nview\n \nof\n \nusers\n \nor\n \nbusinesses.\n  8.  Natural  Language  Data  Exploration:  Users  will  que

In [ ]:
import os
from google.colab import userdata

GROQ_API_KEY=userdata.get("GROQ_API_KEY")

print(GROQ_API_KEY)

In [ ]:
!pip install langchain-groq
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.0 MB/s eta 0:00:00


In [ ]:
class GroqLLM:
  def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
    """
    Initialize Groq LLM
    Args:
    model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
    api_key: Groq API key (or set GROQ_API_KEY environment variable)
    """

    self.model_name = model_name
    self.api_key = api_key

    if not self.api_key:
      raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")

    self.llm = ChatGroq(
        model_name=self.model_name,
        api_key=self.api_key,
        temperature=0.1,
        max_tokens=1024
    )

    print(f"Initialized Groq LLM with model: {self.model_name}")


    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context

        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length

        Returns:
            Generated response string
        """

        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables = ["context","question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.
                      Context:
                      {context}
                      Question: {question}
                    Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )

        formatted_prompt = prompt_template.format(context=context, question=query)

        # Generate response
        messages = [
            HumanMessage(content=formatted_prompt)
        ]

        response = self.llm.invoke(messages)

        return response.content


    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting

        Args:
            query: User question
            context: Retrieved context

        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

            Question: {query}

            Answer:"""

        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"


In [ ]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM("gemma2-9b-it",GROQ_API_KEY)
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [ ]:
rag_retriever.retrieve("What is AI")

Retrieving documents for query: 'What is AI'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents
{'ids': [['doc_d89aa9c6_332', 'doc_454d802c_331', 'doc_55a1ec60_329', 'doc_ad657f31_558', 'doc_47896496_38']], 'embeddings': None, 'documents': [['MD  ARBAAZ  KHAN                                                                Generative  AI  Interview  Questions   91 \n \n 4.  Creative  Collaboration:  AI  will  assist  creatives  by  blending  sketches,  descriptions,  and  \nideas\n \ninto\n \ncohesive\n \noutputs,\n \nboosting\n \nartistic\n \nproductivity.\n  5.  Increased  Accessibility:  Multimodal  AI  will  adapt  interactions  to  individual  abilities,  \nimproving\n \naccessibility\n \nfor\n \npeople\n \nwith\n \ndisabilities.\n  6.  Smarter  Autonomous  Systems:  Enhanced  sensor  integration  will  allow  safer  and  more  \nprecise\n \nnavigation\n \nfor\n \nautonomous\n \nvehicles\n \nand\n \ndrones.\n  7.  Data  Integration  Across  Platforms:  AI  will  reduce  data  silos,  combining  information  from  

[{'id': 'doc_d89aa9c6_332',
  'content': 'MD  ARBAAZ  KHAN                                                                Generative  AI  Interview  Questions   91 \n \n 4.  Creative  Collaboration:  AI  will  assist  creatives  by  blending  sketches,  descriptions,  and  \nideas\n \ninto\n \ncohesive\n \noutputs,\n \nboosting\n \nartistic\n \nproductivity.\n  5.  Increased  Accessibility:  Multimodal  AI  will  adapt  interactions  to  individual  abilities,  \nimproving\n \naccessibility\n \nfor\n \npeople\n \nwith\n \ndisabilities.\n  6.  Smarter  Autonomous  Systems:  Enhanced  sensor  integration  will  allow  safer  and  more  \nprecise\n \nnavigation\n \nfor\n \nautonomous\n \nvehicles\n \nand\n \ndrones.\n  7.  Data  Integration  Across  Platforms:  AI  will  reduce  data  silos,  combining  information  from  \nvarious\n \nsources\n \nfor\n \na\n \nmore\n \nholistic\n \nview\n \nof\n \nusers\n \nor\n \nbusinesses.\n  8.  Natural  Language  Data  Exploration:  Users  will  que

 {
  'id': 'doc_454d802c_331',
  'content': 'AI\n \nand\n \ntheir\n \npotential\n \nimpacts: 1.  Deeper  Context  Understanding:  AI  will  interpret  emotions,  tone,  and  gestures,  making  \ninteractions\n \nmore\n \nempathetic\n \nand\n \ncontext-aware.\n  2.  Uniﬁed  AI  Assistants:  Consistent,  personalized  AI  experiences  across  devices,  offering  \nseamless\n \nassistance\n \nin\n \nvarious\n \nsettings.\n  3.  AR/VR  Integration:  Multimodal  AI  will  enhance  immersive  experiences,  enabling  natural  \ninteractions\n \nwithin\n \naugmented\n \nand\n \nvirtual\n \nenvironments.',
  
  'metadata': {
   
   'source_file': 'gen-ai-interview-question-answer.pdf',
   'page_label': '91',
   'title': 'gen ai interview',
   'content_length': 518,
   'total_pages': 187,
   'doc_index': 331,
   'producer': 'Skia/PDF m133 Google Docs Renderer',
   'file_type': 'pdf',
   'creationdate': '',
   'page': 90,
   'creator': 'PyPDF',
   'source': '/content/data/gen-ai-interview-question-answer.pdf'},
  'similarity_score': 0.09487158060073853,
  'distance': 0.9051284193992615,
  'rank': 2
 }
  
  }

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model_name="gemma2-9b-it",
    api_key=GROQ_API_KEY,
    temperature=0.1,
    max_tokens=1024
)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
  results = retriever.retrieve(query,top_k=top_k)
  context = "\n\n".join([doc['content'] for doc in results]) if results else ""
  if not context:
    return "No relevant context found to answer the question."

  ## generate the answwer using GROQ LLM
  prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""

  response=llm.invoke([prompt.format(context=context,query=query)])
  return response.content


In [ ]:
answer=rag_simple("What is AI?",rag_retriever,llm)
answer

Retrieving documents for query: 'What is AI?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents
{'ids': [['doc_d89aa9c6_332', 'doc_454d802c_331', 'doc_55a1ec60_329']], 'embeddings': None, 'documents': [['MD  ARBAAZ  KHAN                                                                Generative  AI  Interview  Questions   91 \n \n 4.  Creative  Collaboration:  AI  will  assist  creatives  by  blending  sketches,  descriptions,  and  \nideas\n \ninto\n \ncohesive\n \noutputs,\n \nboosting\n \nartistic\n \nproductivity.\n  5.  Increased  Accessibility:  Multimodal  AI  will  adapt  interactions  to  individual  abilities,  \nimproving\n \naccessibility\n \nfor\n \npeople\n \nwith\n \ndisabilities.\n  6.  Smarter  Autonomous  Systems:  Enhanced  sensor  integration  will  allow  safer  and  more  \nprecise\n \nnavigation\n \nfor\n \nautonomous\n \nvehicles\n \nand\n \ndrones.\n  7.  Data  Integration  Across  Platforms:  AI  will  reduce  data  silos,  combining  information  from  \nvarious\n \nsources\n \nfor\n \na\n \

BadRequestError: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
  """
  RAG pipeline with extra features:
  - Returns answer, sources, confidence score, and optionally full context.
  """

  results = retriever.retrieve(query, top_k=top_k, score_threshold = min_score)
  if not results:
    return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
  context = "\n\n".join([doc['content'] for doc in results]) if results else ""
  sources = [{
      'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
      'page': doc['metadata'].get('page', 'unknown'),
      'score': doc['similarity_score'],
      'preview': doc['content'][:300] + '...'
  } for doc in results]
  confidence = max([doc['similarity_score'] for doc in results])


  # Generate answer
  prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
  formatted_prompt = prompt.format(context=context, query=query)


  response = llm.invoke([formatted_prompt])

  output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
  if return_context:
    output['context'] = context
  return output

# Example usage:
result = rag_advanced("What is legal and compliance policy", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Retrieving documents for query: 'What is legal and compliance policy'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents
{'ids': [['doc_a43125d0_4', 'doc_5433b7e2_10', 'doc_af939c63_7']], 'embeddings': None, 'documents': [['High\nLegal and compliance\nWhat audit rights exist in the DPA?\nBlueLeaf Retail\nMSA, pricing, support SLA\nMedium\nProcurement\nWhich clause controls renewal\npricing?\nCityRide Mobility\nOps reports, incident logs,\nSOPs\nMedium\nOperations\nWhich depot had repeated battery\nincidents?\nCaption: Ownership map showing how client admins, legal teams, audit teams, document storage, vector storage, and the RAG service interact.', 'Complex RAG Parsing Sample - synthetic document\nPage 5\n4. Simple Table: Policy Rules\nThe following table is intentionally simple. It should be correctly extracted by most PDF table parsers. It tests basic row and\ncolumn detection, short text values, and numeric values.\nPolicy Area\nRule\nOwner\nReview Cycle\nRefunds\nRefund requests must be raised within 7 days of\npurchase.\nCustomer Suppor

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents
{'ids': [['doc_cd1f8c03_202', 'doc_d6a3edf1_201', 'doc_e15f0618_401']], 'embeddings': None, 'documents': [["reconstruct\n \nit\n \ncorrectly.\n  \nQ12.  Why  is  multi-head  attention  needed? Multihead  attention  is  a  technique  used  in  transformer  models  that  enables  them  to  learn  \nand\n \nfocus\n \non\n \ndifferent\n \nparts\n \nof\n \na\n \nsequence\n \nsimultaneously,\n \nwhich\n \nimproves\n \nthe\n \nmodel's\n \nability\n \nto\n \ncapture\n \ncomplex\n \ndependencies\n \nin\n \nthe\n \ndata.\n \n  \nHere's\n \nwhy\n \nit's\n \nneeded\n \nand\n \nbeneﬁcial:\n ➢  Capturing  Diverse  Relationships  ➢  Mitigating  Information  Bottlenecks  ➢  Learning  Contextual  Nuances", 'in\n \nlearning\n \ndependencies.\n ➔  Example:  In  a  sentence,  "The  cat  sat  on  the  mat,"  different  word  order  permutations  \nmight\n \nbe\n \nused,\n \nsuch\n \nas\n \n"on\n \nthe\n \ncat\n \nmat\n \nsat."\n  5.  Denoising

BadRequestError: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}